# Evaluate — Full Pipeline Comparison

Compares crop-based pipelines and YOLO against CVAT ground truth.

**Run order:** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 (metrics) → 10 (viz) → 11 (browse)

## Cell 1 — Environment  ← edit your path here

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    import zipfile, os
    DRIVE_ROOT  = Path('/content/drive/MyDrive')
    ZIP_PATH    = DRIVE_ROOT / 'pollinator-colab.zip'
    EXTRACT_TO  = Path('/content/pollinator-colab')

    # Re-extract if session was reset
    if not EXTRACT_TO.exists():
        print(f'Extracting {ZIP_PATH.name} to /content/ ...')
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall('/content/')
        print('✓ Extracted')
    else:
        print('✓ Already extracted')

    BASE_DIR   = EXTRACT_TO
    DRIVE_BASE = DRIVE_ROOT / 'pollinator-colab'
else:
    BASE_DIR   = Path('/Users/lianshi/Downloads/bachelor thesis'
                      '/automated-ecological-image-analysis'
                      '/ml-pipelines/notebooks/pollinator-classification')
    DRIVE_BASE = BASE_DIR

IMAGE_ROOT        = BASE_DIR  / 'Insects_images' / 'e2e_evaluation_images'
GT_ANN_ROOT       = BASE_DIR  / 'Insects_images' / 'e2e_yolo_annotations'
MODEL_DIR         = BASE_DIR  / 'models'
CROP_RESULTS_ROOT = (DRIVE_BASE if IN_COLAB else BASE_DIR) / 'Insects_images' / 'crop_results'
YOLO_RESULTS_ROOT = (DRIVE_BASE if IN_COLAB else BASE_DIR) / 'Insects_images' / 'yolo_results'
EVAL_DIR          = BASE_DIR  / 'evaluation'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print(f'Env              : {"Colab" if IN_COLAB else "Local"}')
print(f'IMAGE_ROOT       : {IMAGE_ROOT}  exists={IMAGE_ROOT.exists()}')
print(f'GT_ANN_ROOT      : {GT_ANN_ROOT}  exists={GT_ANN_ROOT.exists()}')
print(f'CROP_RESULTS_ROOT: {CROP_RESULTS_ROOT}  exists={CROP_RESULTS_ROOT.exists()}')
print(f'YOLO_RESULTS_ROOT: {YOLO_RESULTS_ROOT}  exists={YOLO_RESULTS_ROOT.exists()}')


## Cell 2 — Evaluation config  ← edit here

In [ ]:
# ── Which runs to evaluate ──────────────────────────────────────
CROP_RUNS = {
    'run_01': CROP_RESULTS_ROOT / 'run_01',
    # 'run_02_lm_off': CROP_RESULTS_ROOT / 'run_02_lm_off',
}
YOLO_RUNS = {
    'yolo_run_01': YOLO_RESULTS_ROOT / 'yolo_run_01',
}

GT_CLASSES   = ['bumblebee', 'fly', 'butterfly', 'other']
STRIP_HEIGHT = 120   # px removed from bottom (Wingscapes OSD bar)

print('Runs to evaluate:')
for name, path in {**CROP_RUNS, **YOLO_RUNS}.items():
    print(f'  {name}: exists={path.exists()}')


## Cell 3 — Imports

In [ ]:
import csv, json as _json
from pathlib import Path
from collections import defaultdict
import numpy as np, cv2
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
print('✓ Imports done')


## Cell 4 — Load ground truth

Reads CVAT YOLO 1.1 annotations.
Uses **original image dimensions** for coordinate conversion (not stripped height).

In [ ]:
def load_gt(gt_ann_root, image_root, gt_classes, strip_height=120):
    print('Loading ground truth...')
    gt = {}
    for cam_dir in sorted(Path(gt_ann_root).iterdir()):
        if not cam_dir.is_dir(): continue
        lbl_dir = cam_dir / 'obj_train_data'
        img_dir = Path(image_root) / cam_dir.name
        if not lbl_dir.exists() or not img_dir.exists(): continue
        names_f = cam_dir / 'obj.names'
        cls_names = ([l.strip() for l in names_f.read_text().splitlines() if l.strip()]
                     if names_f.exists() else gt_classes)
        n_cam = 0
        for txt in sorted(lbl_dir.glob('*.txt')):
            img_p = None
            for ext in ('.JPG','.jpg','.jpeg','.png'):
                cand = img_dir / (txt.stem + ext)
                if cand.exists(): img_p = cand; break
            if img_p is None: continue
            img = cv2.imread(str(img_p))
            if img is None: continue
            H_orig, W = img.shape[:2]  # always use ORIGINAL dims
            boxes = []
            for line in txt.read_text().strip().splitlines():
                parts = line.strip().split()
                if len(parts) < 5: continue
                try: ci,cx,cy,bw,bh = int(parts[0]),*[float(x) for x in parts[1:5]]
                except ValueError: continue
                cname = cls_names[ci] if ci < len(cls_names) else str(ci)
                y1 = (cy - bh/2) * H_orig
                y2 = (cy + bh/2) * H_orig
                # skip boxes entirely in OSD strip
                if strip_height > 0 and y1 >= (H_orig - strip_height): continue
                y2 = min(y2, H_orig - strip_height)
                boxes.append({'cls': cname,
                              'x1': (cx-bw/2)*W, 'y1': y1,
                              'x2': (cx+bw/2)*W, 'y2': y2})
            gt[str(img_p)] = boxes
            n_cam += len(boxes)
        if n_cam: print(f'  {cam_dir.name}: {n_cam} annotations')
    n_total = sum(len(v) for v in gt.values())
    cls_cnt = {}
    for boxes in gt.values():
        for b in boxes: cls_cnt[b['cls']] = cls_cnt.get(b['cls'],0)+1
    print(f'\n✓ GT: {len(gt)} images  {n_total} annotations')
    for c,n in sorted(cls_cnt.items()): print(f'  {c:15}: {n}')
    return gt

gt = load_gt(GT_ANN_ROOT, IMAGE_ROOT, GT_CLASSES, STRIP_HEIGHT)
gt_with_boxes = {k:v for k,v in gt.items() if len(v) > 0}
print(f'Images with annotations: {len(gt_with_boxes)}')


## Cell 5 — Load inference results

Loads all crop and YOLO results.
**Crop results**: every candidate bbox row (both insect AND background rejected).
**YOLO results**: every detection from yolo_results.csv.

In [ ]:
def detect_pipelines(csv_path):
    with open(csv_path, newline='') as f:
        fields = csv.DictReader(f).fieldnames or []
    return sorted({f.split('__')[0] for f in fields
                   if '__binary_label' in f or '__pollinator_type' in f})

def load_crop_run(run_path, image_root):
    run_path = Path(run_path)
    cfg_file = run_path / 'run_config.json'
    run_cfg  = _json.loads(cfg_file.read_text()) if cfg_file.exists() else {}
    rows = []; pipe_names = []
    for csv_path in sorted(run_path.rglob('results.csv')):
        cam_name = csv_path.parent.name
        img_dir  = Path(image_root) / cam_name
        if not pipe_names:
            pipe_names = detect_pipelines(csv_path)
        with open(csv_path, newline='') as f:
            for row in csv.DictReader(f):
                img_name = row.get('image_name', '')
                # Build full path from camera_folder + image_name
                img_p = str(img_dir / img_name)
                if not Path(img_p).exists():
                    stem = Path(img_name).stem
                    for ext in ('.JPG','.jpg','.jpeg'):
                        cand = str(img_dir / (stem+ext))
                        if Path(cand).exists(): img_p=cand; break
                row['_img_path'] = img_p
                row['_cam']      = cam_name
                rows.append(row)
    pre = run_cfg.get('preprocess', {})
    print(f'  Pipelines : {pipe_names}')
    print(f'  Rows      : {len(rows)}')
    print(f'  Config    : large_motion={pre.get("enable_large_motion","?")}  '
          f'darker_threshold={pre.get("darker_threshold","?")}')
    return rows, pipe_names, run_cfg

def load_yolo_run(run_path, image_root):
    run_path = Path(run_path)
    cfg_file = run_path / 'run_config.json'
    run_cfg  = _json.loads(cfg_file.read_text()) if cfg_file.exists() else {}
    rows = []
    for csv_path in sorted(run_path.rglob('yolo_results.csv')):
        cam_name = csv_path.parent.name
        img_dir  = Path(image_root) / cam_name
        with open(csv_path, newline='') as f:
            for row in csv.DictReader(f):
                img_name = row.get('image_name', '')
                img_p = str(img_dir / img_name)
                if not Path(img_p).exists():
                    stem = Path(img_name).stem
                    for ext in ('.JPG','.jpg','.jpeg'):
                        cand = str(img_dir / (stem+ext))
                        if Path(cand).exists(): img_p=cand; break
                row['_img_path'] = img_p
                row['_cam']      = cam_name
                rows.append(row)
    print(f'  YOLO detections: {len(rows)}')
    return rows, run_cfg

print('Loading crop runs...')
crop_run_data = {}
for name, path in CROP_RUNS.items():
    print(f'\n  {name}:')
    rows, pipes, cfg = load_crop_run(path, IMAGE_ROOT)
    crop_run_data[name] = {'rows':rows,'pipes':pipes,'config':cfg}

print('\nLoading YOLO runs...')
yolo_run_data = {}
for name, path in YOLO_RUNS.items():
    print(f'\n  {name}:')
    rows, cfg = load_yolo_run(path, IMAGE_ROOT)
    yolo_run_data[name] = {'rows':rows,'config':cfg}

print('\n✓ All runs loaded.')


## Cell 6 — Matching + metrics functions

**Do not edit.** Defines all evaluation logic.

**`center_match`**: matches predictions to GT using three criteria (any one sufficient):
1. GT center falls inside pred bbox
2. Pred center falls inside GT bbox
3. Overlap area / GT area ≥ 20%

**`evaluate_one_pipeline`**: for one pipeline computes:
- **TP** — bbox matched GT (any criterion above) + correct class
- **FP** — bbox with no matching GT (false alarm)
- **FN** — GT bbox with no matching prediction (missed insect)
- **FN breakdown**:
  - `fn_detected_as_bg` — pipeline HAD a bbox near this insect but classified it as background
  - `fn_not_detected` — no bbox at all near this insect (frame-diff completely missed it)
- **bg_rejected** — total candidate crops classified as background


In [ ]:
def bbox_overlap_ratio(p, g):
    ix1=max(p['x1'],g['x1']); iy1=max(p['y1'],g['y1'])
    ix2=min(p['x2'],g['x2']); iy2=min(p['y2'],g['y2'])
    iw=max(0,ix2-ix1); ih=max(0,iy2-iy1)
    gt_area=max(1,(g['x2']-g['x1'])*(g['y2']-g['y1']))
    return iw*ih/gt_area

def center_match(pred_boxes, gt_boxes, overlap_thresh=0.20):
    """Three criteria: GT center in pred, pred center in GT, or 20% overlap."""
    def inside(px,py,x1,y1,x2,y2): return x1<=px<=x2 and y1<=py<=y2
    mp=set(); mg=set(); pairs=[]
    for gi,g in enumerate(gt_boxes):
        gcx=(g['x1']+g['x2'])/2; gcy=(g['y1']+g['y2'])/2
        for pi,p in enumerate(pred_boxes):
            if pi in mp: continue
            pcx=(p['x1']+p['x2'])/2; pcy=(p['y1']+p['y2'])/2
            if (inside(gcx,gcy,p['x1'],p['y1'],p['x2'],p['y2']) or
                inside(pcx,pcy,g['x1'],g['y1'],g['x2'],g['y2']) or
                bbox_overlap_ratio(p,g)>=overlap_thresh):
                pairs.append((pi,gi)); mp.add(pi); mg.add(gi); break
    return pairs,[i for i in range(len(pred_boxes)) if i not in mp],\
                 [i for i in range(len(gt_boxes)) if i not in mg]

def evaluate_one_pipeline(preds_by_img, gt, classes, label):
    """
    Two-level evaluation:
    1. Detection: did pipeline find a bbox near the insect? (class-agnostic)
    2. Classification: was the class correct? (among detected)

    Also tracks:
    - fn_detected_as_bg: GT insects that WERE detected but classified as background
    - fn_not_detected:   GT insects that had NO bbox near them at all
    """
    rows_out=[]; n_bg_rejected=0
    det_tp=det_fp=det_fn=0
    cls_correct=0; cls_wrong=0
    tp_c=defaultdict(int); fp_c=defaultdict(int); fn_c=defaultdict(int)
    cls_confusion=defaultdict(lambda: defaultdict(int))

    # FN breakdown
    fn_detected_as_bg=0   # had a bbox but was rejected as background
    fn_not_detected=0     # no bbox at all near this GT insect

    for img_p, gt_boxes in gt.items():
        all_preds = preds_by_img.get(img_p, [])
        insect    = [p for p in all_preds if not p.get('is_bg')]
        rejected  = [p for p in all_preds if p.get('is_bg')]
        n_bg_rejected += len(rejected)

        # Match insect predictions to GT
        pairs, unp, ung = center_match(insect, gt_boxes)

        for pi,gi in pairs:
            det_tp+=1
            pc=insect[pi]['cls']; gc=gt_boxes[gi]['cls']
            correct=(pc==gc)
            if correct: cls_correct+=1; tp_c[gc]+=1
            else: cls_wrong+=1; fp_c[pc]+=1; fn_c[gc]+=1
            cls_confusion[gc][pc]+=1
            rows_out.append({'pipeline':label,'img':img_p,'match':'tp',
                             'pred_cls':pc,'gt_cls':gc,
                             'conf':insect[pi]['conf'],'correct_cls':correct})
        for pi in unp:
            det_fp+=1; fp_c[insect[pi]['cls']]+=1
            rows_out.append({'pipeline':label,'img':img_p,'match':'fp',
                             'pred_cls':insect[pi]['cls'],'gt_cls':'',
                             'conf':insect[pi]['conf'],'correct_cls':False})

        # For each unmatched GT, check if a REJECTED bbox covers it
        for gi in ung:
            det_fn+=1; fn_c[gt_boxes[gi]['cls']]+=1
            g = gt_boxes[gi]
            # Check if any bg_rejected bbox overlaps this GT
            covered_by_bg = any(
                bbox_overlap_ratio(r, g) >= 0.20 or
                bbox_overlap_ratio(g, r) >= 0.20
                for r in rejected
            )
            if covered_by_bg:
                fn_detected_as_bg+=1
                match_type='fn_detected_as_bg'
            else:
                fn_not_detected+=1
                match_type='fn_not_detected'
            rows_out.append({'pipeline':label,'img':img_p,'match':match_type,
                             'pred_cls':'bg','gt_cls':gt_boxes[gi]['cls'],
                             'conf':0.0,'correct_cls':False})

    det_prec=det_tp/max(1,det_tp+det_fp)
    det_rec =det_tp/max(1,det_tp+det_fn)
    det_f1  =2*det_prec*det_rec/max(1e-8,det_prec+det_rec)
    cls_acc =cls_correct/max(1,det_tp)

    print(f'  [Detection]      P={det_prec:.3f}  R={det_rec:.3f}  F1={det_f1:.3f}  '
          f'TP={det_tp}  FP={det_fp}  FN={det_fn}  bg_rejected={n_bg_rejected}')
    print(f'  [FN breakdown]   detected_as_bg={fn_detected_as_bg}  '
          f'not_detected={fn_not_detected}  '
          f'({100*fn_detected_as_bg/max(1,det_fn):.1f}% were detected but rejected)')
    print(f'  [Classification] accuracy={cls_acc:.3f}  '
          f'correct={cls_correct}  wrong={cls_wrong}  (of {det_tp} detected)')
    print(f'  Per-class:')
    for c in classes:
        if tp_c[c] or fp_c.get(c) or fn_c[c]:
            print(f'    {c:15} TP={tp_c[c]:>4}  FP={fp_c.get(c,0):>4}  FN={fn_c[c]:>4}')

    return {'det_precision':det_prec,'det_recall':det_rec,'det_f1':det_f1,
            'det_tp':det_tp,'det_fp':det_fp,'det_fn':det_fn,
            'fn_detected_as_bg':fn_detected_as_bg,
            'fn_not_detected':fn_not_detected,
            'cls_accuracy':cls_acc,'cls_correct':cls_correct,'cls_wrong':cls_wrong,
            'n_bg_rejected':n_bg_rejected,
            'tp_c':dict(tp_c),'fp_c':dict(fp_c),'fn_c':dict(fn_c),
            'cls_confusion':dict(cls_confusion),
            'rows':rows_out}


## Cell 7 — Build prediction index

Organises all predictions by image path for efficient matching.

For crop pipelines: every candidate bbox is included, both insect and background predictions.
`is_bg=True` means the pipeline classified this bbox as background.

For YOLO: every detection row from `yolo_results.csv`.


In [ ]:
def build_crop_index(rows, pipe_name):
    """Build {img_path -> list of pred dicts} for one pipeline."""
    idx = defaultdict(list)
    for r in rows:
        if r.get('pollinator_detected') not in ('yes',): continue
        try:
            x=int(r['bbox_x']); y=int(r['bbox_y'])
            w=int(r['bbox_w']); h=int(r['bbox_h'])
        except (ValueError,KeyError): continue
        p = pipe_name + '__'
        bl  = r.get(p+'binary_label', '')
        pt  = r.get(p+'pollinator_type', '')
        # Determine if this bbox is insect or background for this pipeline
        is_bg = (bl == 'background') or (pt == 'background') or \
                (not bl and not pt)
        pred_cls = pt if (pt and pt != 'background') else (bl if bl == 'insect' else 'background')
        try: conf = float(r.get(p+'group_conf') or r.get(p+'binary_conf') or 0)
        except: conf = 0.0
        idx[r['_img_path']].append({
            'x1':x,'y1':y,'x2':x+w,'y2':y+h,
            'cls':pred_cls,'conf':conf,'is_bg':is_bg
        })
    return idx

def build_yolo_index(rows):
    idx = defaultdict(list)
    for r in rows:
        try:
            x=int(r['bbox_x']); y=int(r['bbox_y'])
            w=int(r['bbox_w']); h=int(r['bbox_h'])
            conf=float(r.get('confidence',0))
        except (ValueError,KeyError): continue
        idx[r['_img_path']].append({
            'x1':x,'y1':y,'x2':x+w,'y2':y+h,
            'cls':r.get('class_name',''),'conf':conf,'is_bg':False
        })
    return idx

# Build all indexes
pred_indexes = {}  # label -> {img_path -> [preds]}

for run_name, run_data in crop_run_data.items():
    for pipe_name in run_data['pipes']:
        label = f'{run_name}/{pipe_name}'
        pred_indexes[label] = build_crop_index(run_data['rows'], pipe_name)
        n = sum(len(v) for v in pred_indexes[label].values())
        n_ins = sum(1 for v in pred_indexes[label].values() for p in v if not p['is_bg'])
        n_bg  = sum(1 for v in pred_indexes[label].values() for p in v if p['is_bg'])
        print(f'  {label}: {n} total  insect={n_ins}  background={n_bg}')

for run_name, run_data in yolo_run_data.items():
    label = run_name
    pred_indexes[label] = build_yolo_index(run_data['rows'])
    n = sum(len(v) for v in pred_indexes[label].values())
    print(f'  {label}: {n} detections')

print(f'\n✓ Prediction indexes built for {len(pred_indexes)} pipelines.')


## Cell 8 — Run evaluation  ← main evaluation cell

Evaluates every pipeline against GT. Prints results as it goes.


In [ ]:
all_results = {}

for label, preds_by_img in pred_indexes.items():
    print(f'\n=== {label} ===')
    all_results[label] = evaluate_one_pipeline(
        preds_by_img, gt, GT_CLASSES, label)

print(f'\n✓ Evaluated {len(all_results)} pipelines.')


## Cell 9 — Results table + confusion matrix + save

Prints:
1. **Overall detection metrics** (class-agnostic P/R/F1) with FN breakdown
2. **Per-class P/R/F1**
3. **Classification accuracy** among detected insects
4. **Confusion matrix** (GT class vs predicted class, including bg_rejected and missed)

Saves `eval_results.csv` and `summary.json` to `evaluation/`.


In [ ]:
import csv as _csv
import json as _json
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict

# ── Overall detection summary ─────────────────────────────────────
print('\n' + '='*80)
print(f'{"Pipeline":40}  {"P":>7}  {"R":>7}  {"F1":>7}  {"TP":>5}  {"FP":>5}  {"FN":>5}  {"BG_rej":>7}')
print('='*80)
for label, r in all_results.items():
    print(f'{label:40}  {r["det_precision"]:>7.3f}  {r["det_recall"]:>7.3f}  '
          f'{r["det_f1"]:>7.3f}  {r["det_tp"]:>5}  {r["det_fp"]:>5}  '
          f'{r["det_fn"]:>5}  {r["n_bg_rejected"]:>7}')

# ── Per-class detection metrics ───────────────────────────────────
print('\n' + '='*80)
print('Per-class Detection Metrics')
print('='*80)
for label, r in all_results.items():
    print(f'\n  {label}')
    print(f'  {"Class":15}  {"P":>7}  {"R":>7}  {"F1":>7}  {"TP":>5}  {"FP":>5}  {"FN":>5}')
    print(f'  {"-"*60}')
    for c in GT_CLASSES:
        tp = r["tp_c"].get(c, 0)
        fp = r["fp_c"].get(c, 0)
        fn = r["fn_c"].get(c, 0)
        p  = tp / max(1, tp+fp)
        rc = tp / max(1, tp+fn)
        f1 = 2*p*rc / max(1e-8, p+rc)
        print(f'  {c:15}  {p:>7.3f}  {rc:>7.3f}  {f1:>7.3f}  {tp:>5}  {fp:>5}  {fn:>5}')

# ── Classification accuracy ───────────────────────────────────────
print('\n' + '='*80)
print('Classification Accuracy (among detected insects)')
print('='*80)
print(f'{"Pipeline":40}  {"Accuracy":>9}  {"Correct":>8}  {"Wrong":>7}')
print('-'*70)
for label, r in all_results.items():
    print(f'{label:40}  {r["cls_accuracy"]:>9.3f}  '
          f'{r["cls_correct"]:>8}  {r["cls_wrong"]:>7}')

# ── Confusion matrices ────────────────────────────────────────────
print('\n' + '='*80)
print('Confusion Matrices (rows=GT, cols=Predicted)')
print('(bg_rejected=detected but classified as background, missed=not detected)')
print('='*80)

for label, r in all_results.items():
    print(f'\n  {label}')
    rows = r['rows']
    # Build matrix: gt_cls -> pred_cls -> count
    matrix = defaultdict(lambda: defaultdict(int))
    missed = defaultdict(int)   # gt_cls -> count of FN (not detected at all)
    bg_rej = defaultdict(int)   # gt_cls -> count rejected as background

    for row in rows:
        if row['match'] == 'tp':
            matrix[row['gt_cls']][row['pred_cls']] += 1
        elif row['match'] == 'fn':
            missed[row['gt_cls']] += 1

    # bg_rejected comes from rows marked 'rejected_as_bg' if available
    # otherwise count from n_bg_rejected per class (approximate)
    # Use tp_c + fn_c to infer
    for c in GT_CLASSES:
        bg_rej[c] = r['fn_c'].get(c, 0) - missed[c]
        if bg_rej[c] < 0: bg_rej[c] = 0

    cols = GT_CLASSES + ['bg_rejected', 'missed']
    col_w = 12
    gt_pred = "GT \\ Pred"
    header = f'  {gt_pred:15}' + ''.join(f'{c:>{col_w}}' for c in cols)
    print(header)
    print('  ' + '-'*(15 + col_w*len(cols)))
    for gt_c in GT_CLASSES:
        row_str = f'  {gt_c:15}'
        for pred_c in GT_CLASSES:
            row_str += f'{matrix[gt_c][pred_c]:>{col_w}}'
        row_str += f'{bg_rej[gt_c]:>{col_w}}'
        row_str += f'{missed[gt_c]:>{col_w}}'
        print(row_str)

# ── Save summary ──────────────────────────────────────────────────
all_rows = [row for r in all_results.values() for row in r['rows']]
with open(EVAL_DIR/'eval_results.csv', 'w', newline='') as fh:
    w = _csv.DictWriter(fh, fieldnames=
        ['pipeline','img','match','pred_cls','gt_cls','conf','correct_cls'])
    w.writeheader(); w.writerows(all_rows)

def _ser(o):
    if isinstance(o, dict): return {str(k):_ser(v) for k,v in o.items()}
    if isinstance(o, (list,tuple)): return [_ser(x) for x in o]
    if isinstance(o, (np.integer, np.floating)): return o.item()
    return o

summary = {label:{k:v for k,v in r.items() if k!='rows'}
           for label,r in all_results.items()}
(EVAL_DIR/'summary.json').write_text(_json.dumps(_ser(summary), indent=2))
print(f'\n✓ Saved to {EVAL_DIR}')


## Cell 10 — Confidence Threshold Analysis

For each pipeline, shows how P/R/F1 change at different confidence thresholds.

**Goal:** find the threshold that achieves target recall (≥0.8) with best precision.
Since the goal is not to miss insects, we prioritise recall.

Saves `threshold_analysis.png` to `evaluation/`.


In [ ]:
# ── Threshold analysis ────────────────────────────────────────────
RECALL_TARGETS = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]

print('\n' + '='*75)
print('Threshold Analysis — Recall-constrained')
print('(find best threshold for each recall target)')
print('='*75)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = plt.cm.tab10.colors

for ci, (label, r) in enumerate(all_results.items()):
    color = colors[ci % len(colors)]

    # Get all (conf, is_tp) pairs from rows
    dets = sorted(
        [(row['conf'], row['match'] == 'tp')
         for row in r['rows'] if row['match'] in ('tp', 'fp')],
        key=lambda x: -x[0])
    if not dets: continue

    n_gt = r['det_tp'] + r['det_fn']
    thresholds=[]; precisions=[]; recalls=[]; f1s=[]
    tp_ = fp_ = 0

    for conf, is_tp in dets:
        if is_tp: tp_ += 1
        else: fp_ += 1
        p  = tp_ / max(1, tp_+fp_)
        rc = tp_ / max(1, n_gt)
        f1 = 2*p*rc / max(1e-8, p+rc)
        thresholds.append(conf)
        precisions.append(p)
        recalls.append(rc)
        f1s.append(f1)

    # PR curve
    axes[0].plot(recalls, precisions, color=color,
                 label=f'{label} (F1={r["det_f1"]:.3f})', lw=2)

    # Threshold vs Recall/Precision
    axes[1].plot(thresholds, recalls, color=color, ls='-', lw=2,
                 label=f'{label} recall')
    axes[1].plot(thresholds, precisions, color=color, ls='--', lw=1,
                 alpha=0.6)

    # Find best threshold for each recall target
    print(f'\n  {label}')
    print(f'  {"Recall target":15}  {"Threshold":>10}  {"Precision":>10}  {"F1":>8}')
    print(f'  {"-"*50}')
    for target in RECALL_TARGETS:
        # Find highest threshold that achieves this recall
        best_thr = best_p = best_f1 = None
        for thr, p, rc, f1 in zip(thresholds, precisions, recalls, f1s):
            if rc >= target:
                if best_thr is None or thr > best_thr:
                    best_thr = thr; best_p = p; best_f1 = f1
        if best_thr is not None:
            print(f'  R≥{target:.2f}          {best_thr:>10.3f}  {best_p:>10.3f}  {best_f1:>8.3f}')
        else:
            print(f'  R≥{target:.2f}          {"N/A":>10}  {"N/A":>10}  {"N/A":>8}')

    # Mark optimal F1 point
    best_idx = int(np.argmax(f1s))
    axes[0].plot(recalls[best_idx], precisions[best_idx],
                 'o', color=color, ms=8)
    axes[0].annotate(f'  thr={thresholds[best_idx]:.2f}',
                     (recalls[best_idx], precisions[best_idx]),
                     fontsize=7, color=color)

axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('PR Curve (● = best F1)')
axes[0].legend(fontsize=7); axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0,1); axes[0].set_ylim(0,1)

axes[1].set_xlabel('Confidence Threshold')
axes[1].set_ylabel('Score')
axes[1].set_title('Recall (—) and Precision (--) vs Threshold')
axes[1].legend(fontsize=7); axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0,1); axes[1].set_ylim(0,1)

# Mark recall=0.8 line
axes[0].axhline(y=0, color='grey', ls=':', alpha=0.5)
axes[1].axhline(y=0.8, color='grey', ls=':', lw=1, label='R=0.8 target')

plt.suptitle('Confidence Threshold Analysis', fontsize=13)
plt.tight_layout()
plt.savefig(EVAL_DIR/'threshold_analysis.png', dpi=150)
print(f'\n✓ Saved threshold_analysis.png')
plt.show()


## Cell 11 — Browse images with all pipeline bboxes

Displays original images with GT + all pipeline bboxes drawn in memory.
**No files are saved.** Uses matplotlib for cross-platform keyboard navigation.

**Keys:** ← → to navigate, q to quit.

Run `%matplotlib tk` in a separate cell first if using locally.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2, numpy as np
from pathlib import Path
from collections import defaultdict

# ── Build display index: img_path -> list of (label,x1,y1,x2,y2,color,thick) ──
display_index = defaultdict(list)

COLORS = {
    'GT':             (0,   0.78, 0),      # green
    'two_stage':      (1,   0.39, 0),      # blue-ish orange
    'five_class_eff': (0,   0.55, 1),      # orange
    'five_class_ins': (0.7, 0,   1),       # purple
    'yolo_run_01':    (1,   0,   0),       # red
}
DEFAULT_COLOR = (0.5, 0.5, 0.5)

# GT
for img_p, boxes in gt_with_boxes.items():
    for b in boxes:
        display_index[img_p].append(
            (f'GT:{b["cls"]}', b['x1'],b['y1'],b['x2'],b['y2'],
             COLORS['GT'], 2))

# All pipelines
for label, preds_by_img in pred_indexes.items():
    pipe_key = label.split('/')[-1] if '/' in label else label
    color = COLORS.get(pipe_key, DEFAULT_COLOR)
    for img_p, preds in preds_by_img.items():
        for p in preds:
            is_bg = p.get('is_bg', False)
            thick = 0.5 if is_bg else 2
            lbl   = '' if is_bg else f'{pipe_key}:{p["cls"]}'
            c     = (0.7,0.7,0.7) if is_bg else color
            display_index[img_p].append(
                (lbl, p['x1'],p['y1'],p['x2'],p['y2'], c, thick))

img_list = [p for p in sorted(display_index.keys()) if display_index[p]]
print(f'{len(img_list)} images with detections or GT')
print('Keys: ← → to navigate, q to quit')

idx = [0]
fig, ax = plt.subplots(figsize=(16, 9))
plt.subplots_adjust(top=0.95, bottom=0.02)

def draw(i):
    ax.clear()
    img_p = img_list[i]
    img   = cv2.imread(img_p)
    if img is None: return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    for lbl,x1,y1,x2,y2,color,thick in display_index[img_p]:
        rect = mpatches.Rectangle(
            (x1,y1), x2-x1, y2-y1,
            linewidth=thick, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        if lbl:
            ax.text(x1, max(y1-4,10), lbl,
                    color=color, fontsize=6,
                    bbox=dict(fc='black', alpha=0.4, pad=1, ec='none'))
    name = f'{Path(img_p).parent.name}/{Path(img_p).name}'
    ax.set_title(f'[{i+1}/{len(img_list)}]  {name}', fontsize=9)
    ax.axis('off')
    fig.canvas.draw_idle()

def on_key(event):
    if event.key == 'right':
        idx[0] = min(len(img_list)-1, idx[0]+1); draw(idx[0])
    elif event.key == 'left':
        idx[0] = max(0, idx[0]-1); draw(idx[0])
    elif event.key == 'q':
        plt.close()

fig.canvas.mpl_connect('key_press_event', on_key)
draw(0)
plt.show()


In [ ]:
%matplotlib tk # for interactive viewing, 